# Post-2M Evaluation

Use this notebook after the medium MaskablePPO run finishes. It resolves the trained checkpoint, runs larger holdout evaluations, probes cross-profile generalization, and writes evaluation artifacts under `runs/eval/`.

Recommended order:

1. Run the setup and checkpoint cells.
2. Run the medium holdout evaluation.
3. Run cross-profile evaluations on easy, medium, and hard.
4. Compare against the random baselines and decide whether to move to curriculum training or throughput work first.

In [1]:
from __future__ import annotations

import csv
import json
import os
import re
import sys
from dataclasses import asdict
from pathlib import Path
from typing import Any

import numpy as np

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.random_baseline import EpisodeResult, summarize
from scripts.train_maskable_ppo import evaluate_model, import_training_deps, max_preset_buildings

print(f"repo: {REPO}")

repo: /workspace/ClashAI


## Configuration

For a quick smoke test, lower `MEDIUM_HOLDOUT_EPISODES` and `CROSS_PROFILE_EPISODES` to `5-20`. For the real post-run readout, keep medium at `100-200` episodes.

In [2]:
RUN_ID = "medium_w96_cuda_seed42_20260515_205523"
CHECKPOINT_ROOT = Path("checkpoints/maskable_ppo") / RUN_ID

DEVICE = "cuda"
DETERMINISTIC = True
ARMY_COMPOSITION = {"barbarian": 40, "wall_breaker": 10}
MAX_BUILDINGS = max_preset_buildings()

MEDIUM_HOLDOUT_EPISODES = 200
MEDIUM_HOLDOUT_SEED_START = 100_000

CROSS_PROFILE_EPISODES = 50
CROSS_PROFILE_SEED_START = 200_000
CROSS_PROFILE_NAMES = ["easy", "medium", "hard"]

ARTIFACT_DIR = Path("runs/eval") / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"run: {RUN_ID}")
print(f"checkpoint root: {CHECKPOINT_ROOT}")
print(f"artifacts: {ARTIFACT_DIR}")

run: medium_w96_cuda_seed42_20260515_205523
checkpoint root: checkpoints/maskable_ppo/medium_w96_cuda_seed42_20260515_205523
artifacts: runs/eval/medium_w96_cuda_seed42_20260515_205523


In [3]:
def checkpoint_step(path: Path) -> int:
    match = re.search(r"checkpoint_(\d+)_steps\.zip$", path.name)
    return int(match.group(1)) if match else -1


def resolve_checkpoint(root: Path) -> Path:
    preferred = [root / "final_model.zip", root / "best_model.zip"]
    for path in preferred:
        if path.exists():
            return path
    checkpoints = sorted(root.glob("checkpoint_*_steps.zip"), key=checkpoint_step)
    if checkpoints:
        return checkpoints[-1]
    raise FileNotFoundError(f"no checkpoint found under {root}")


CHECKPOINT_PATH = resolve_checkpoint(CHECKPOINT_ROOT)
print(f"checkpoint: {CHECKPOINT_PATH}")

checkpoint: checkpoints/maskable_ppo/medium_w96_cuda_seed42_20260515_205523/final_model.zip


In [4]:
deps = import_training_deps()
model = deps.MaskablePPO.load(str(CHECKPOINT_PATH), device=DEVICE)
print(f"loaded model on device={DEVICE}")

loaded model on device=cuda


In [5]:
def summary_dict(results: list[EpisodeResult], elapsed: float) -> dict[str, Any]:
    if not results:
        return {"episodes": 0, "elapsed_seconds": elapsed}
    damage = np.array([r.damage_pct for r in results], dtype=np.float64)
    scores = np.array([r.score for r in results], dtype=np.float64)
    stars = np.array([r.stars for r in results], dtype=np.int64)
    steps = np.array([r.steps for r in results], dtype=np.float64)
    ticks = np.array([r.ticks for r in results], dtype=np.float64)
    return {
        "episodes": len(results),
        "elapsed_seconds": float(elapsed),
        "maps_per_min": float(len(results) / max(elapsed, 1e-9) * 60.0),
        "decisions_per_sec": float(steps.sum() / max(elapsed, 1e-9)),
        "sim_ticks_per_sec": float(ticks.sum() / max(elapsed, 1e-9)),
        "damage_mean": float(damage.mean()),
        "damage_median": float(np.median(damage)),
        "damage_min": float(damage.min()),
        "damage_max": float(damage.max()),
        "score_mean": float(scores.mean()),
        "score_median": float(np.median(scores)),
        "stars_mean": float(stars.mean()),
        "stars_0": int((stars == 0).sum()),
        "stars_1": int((stars == 1).sum()),
        "stars_2": int((stars == 2).sum()),
        "stars_3": int((stars == 3).sum()),
        "p_stars_ge_2": float((stars >= 2).mean()),
        "p_damage_ge_50": float((damage >= 0.50).mean()),
        "p_damage_ge_75": float((damage >= 0.75).mean()),
        "p_damage_ge_90": float((damage >= 0.90).mean()),
        "steps_mean": float(steps.mean()),
        "ticks_mean": float(ticks.mean()),
        "terminated": int(sum(r.terminated for r in results)),
        "truncated": int(sum(r.truncated for r in results)),
    }


def write_eval(name: str, results: list[EpisodeResult], elapsed: float) -> dict[str, Any]:
    summary = summary_dict(results, elapsed)
    records = [asdict(result) for result in results]

    json_path = ARTIFACT_DIR / f"{name}.json"
    csv_path = ARTIFACT_DIR / f"{name}.csv"
    with json_path.open("w", encoding="utf-8") as f:
        json.dump({"summary": summary, "episodes": records}, f, indent=2)
    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]) if records else [])
        if records:
            writer.writeheader()
            writer.writerows(records)

    print(f"wrote {json_path}")
    print(f"wrote {csv_path}")
    return summary


def run_eval(name: str, profiles: list[str], seed_start: int, episodes: int) -> dict[str, Any]:
    results, elapsed = evaluate_model(
        model,
        profiles=profiles,
        seed_start=seed_start,
        episodes=episodes,
        max_buildings=MAX_BUILDINGS,
        army_composition=ARMY_COMPOSITION,
        deterministic=DETERMINISTIC,
    )
    print(f"\n[{name}] profiles={profiles} seed_start={seed_start} episodes={episodes}")
    print(summarize(results, elapsed))
    return write_eval(name, results, elapsed)

## Main Medium Holdout

This is the main post-run readout. It should be larger than the in-training `20` episode eval because the early numbers are noisy.

In [6]:
medium_summary = run_eval(
    "medium_holdout",
    profiles=["medium"],
    seed_start=MEDIUM_HOLDOUT_SEED_START,
    episodes=MEDIUM_HOLDOUT_EPISODES,
)
medium_summary


[medium_holdout] profiles=['medium'] seed_start=100000 episodes=200
N=200 elapsed=988.27s maps/min=12.1 decisions/sec=11.9 sim_ticks/sec=118.4
damage mean=0.791 median=0.846 min=0.214 max=1.000 std=0.218
score  mean=2.996 median=2.846 min=0.214 max=4.000
stars  mean=2.205 dist=0:3 1:20 2:110 3:67
steps  mean=58.9 median=56.0 ticks_mean=585.1
ends   terminated=169 truncated=31
damage thresholds >=25%:97.5% >=50%:88.5% >=75%:60.5% >=90%:44.0% >=100%:33.5%
wrote runs/eval/medium_w96_cuda_seed42_20260515_205523/medium_holdout.json
wrote runs/eval/medium_w96_cuda_seed42_20260515_205523/medium_holdout.csv


{'episodes': 200,
 'elapsed_seconds': 988.2749577050563,
 'maps_per_min': 12.142369799459509,
 'decisions_per_sec': 11.922795278919283,
 'sim_ticks_per_sec': 118.40631909942941,
 'damage_mean': 0.7907928348909656,
 'damage_median': 0.8459890965732086,
 'damage_min': 0.2144859813084112,
 'damage_max': 1.0,
 'score_mean': 2.995792834890966,
 'score_median': 2.845989096573209,
 'stars_mean': 2.205,
 'stars_0': 3,
 'stars_1': 20,
 'stars_2': 110,
 'stars_3': 67,
 'p_stars_ge_2': 0.885,
 'p_damage_ge_50': 0.885,
 'p_damage_ge_75': 0.605,
 'p_damage_ge_90': 0.44,
 'steps_mean': 58.915,
 'ticks_mean': 585.09,
 'terminated': 169,
 'truncated': 31}

## Cross-Profile Generalization

The goal here is not to declare hard solved. It tells us whether the policy learned reusable attack behavior or medium-specific quirks.

In [7]:
cross_profile_summaries: dict[str, dict[str, Any]] = {}
for offset, profile in enumerate(CROSS_PROFILE_NAMES):
    cross_profile_summaries[profile] = run_eval(
        f"cross_profile_{profile}",
        profiles=[profile],
        seed_start=CROSS_PROFILE_SEED_START + offset * 10_000,
        episodes=CROSS_PROFILE_EPISODES,
    )
cross_profile_summaries


[cross_profile_easy] profiles=['easy'] seed_start=200000 episodes=50
N=50 elapsed=155.90s maps/min=19.2 decisions/sec=17.5 sim_ticks/sec=173.8
damage mean=0.668 median=0.649 min=0.063 max=1.000 std=0.290
score  mean=2.108 median=2.581 min=0.063 max=4.000
stars  mean=1.440 dist=0:16 1:8 2:14 3:12
steps  mean=54.6 median=54.0 ticks_mean=541.8
ends   terminated=47 truncated=3
damage thresholds >=25%:92.0% >=50%:66.0% >=75%:48.0% >=90%:36.0% >=100%:24.0%
wrote runs/eval/medium_w96_cuda_seed42_20260515_205523/cross_profile_easy.json
wrote runs/eval/medium_w96_cuda_seed42_20260515_205523/cross_profile_easy.csv

[cross_profile_medium] profiles=['medium'] seed_start=210000 episodes=50
N=50 elapsed=248.50s maps/min=12.1 decisions/sec=11.3 sim_ticks/sec=112.4
damage mean=0.807 median=0.934 min=0.304 max=1.000 std=0.229
score  mean=3.107 median=2.934 min=0.420 max=4.000
stars  mean=2.300 dist=0:1 1:6 2:20 3:23
steps  mean=56.3 median=54.0 ticks_mean=558.6
ends   terminated=42 truncated=8
damage 

{'easy': {'episodes': 50,
  'elapsed_seconds': 155.8984916419722,
  'maps_per_min': 19.243290736190268,
  'decisions_per_sec': 17.511394569933145,
  'sim_ticks_per_sec': 173.76691534779812,
  'damage_mean': 0.6679036544850497,
  'damage_median': 0.6492940199335548,
  'damage_min': 0.06279069767441861,
  'damage_max': 1.0,
  'score_mean': 2.10790365448505,
  'score_median': 2.5813953488372094,
  'stars_mean': 1.44,
  'stars_0': 16,
  'stars_1': 8,
  'stars_2': 14,
  'stars_3': 12,
  'p_stars_ge_2': 0.52,
  'p_damage_ge_50': 0.66,
  'p_damage_ge_75': 0.48,
  'p_damage_ge_90': 0.36,
  'steps_mean': 54.6,
  'ticks_mean': 541.8,
  'terminated': 47,
  'truncated': 3},
 'medium': {'episodes': 50,
  'elapsed_seconds': 248.49926047597546,
  'maps_per_min': 12.072470534736402,
  'decisions_per_sec': 11.319953204737834,
  'sim_ticks_per_sec': 112.40274899208573,
  'damage_mean': 0.8068566978193148,
  'damage_median': 0.934190031152648,
  'damage_min': 0.30412772585669784,
  'damage_max': 1.0,
  '

## Baseline Comparison

These random baselines came from the first remote profiling run. Replace them if we rerun baselines with changed env semantics.

In [8]:
random_baselines = {
    "easy": {"damage_mean": 0.932, "score_mean": 3.562, "stars_mean": None, "p_damage_ge_90": None},
    "medium": {"damage_mean": 0.687, "score_mean": 2.101, "stars_mean": 1.414, "p_damage_ge_90": 0.190},
    "hard": {"damage_mean": 0.274, "score_mean": 0.308, "stars_mean": 0.034, "p_damage_ge_90": 0.000},
}

comparison = {}
for profile, summary in cross_profile_summaries.items():
    baseline = random_baselines.get(profile, {})
    comparison[profile] = {
        "policy_score_mean": summary.get("score_mean"),
        "random_score_mean": baseline.get("score_mean"),
        "score_lift": None if baseline.get("score_mean") is None else summary.get("score_mean") - baseline["score_mean"],
        "policy_damage_mean": summary.get("damage_mean"),
        "random_damage_mean": baseline.get("damage_mean"),
        "damage_lift": None if baseline.get("damage_mean") is None else summary.get("damage_mean") - baseline["damage_mean"],
        "policy_stars_mean": summary.get("stars_mean"),
        "random_stars_mean": baseline.get("stars_mean"),
    }

comparison_path = ARTIFACT_DIR / "baseline_comparison.json"
with comparison_path.open("w", encoding="utf-8") as f:
    json.dump(comparison, f, indent=2)
print(f"wrote {comparison_path}")
comparison

wrote runs/eval/medium_w96_cuda_seed42_20260515_205523/baseline_comparison.json


{'easy': {'policy_score_mean': 2.10790365448505,
  'random_score_mean': 3.562,
  'score_lift': -1.4540963455149498,
  'policy_damage_mean': 0.6679036544850497,
  'random_damage_mean': 0.932,
  'damage_lift': -0.2640963455149503,
  'policy_stars_mean': 1.44,
  'random_stars_mean': None},
 'medium': {'policy_score_mean': 3.1068566978193144,
  'random_score_mean': 2.101,
  'score_lift': 1.0058566978193144,
  'policy_damage_mean': 0.8068566978193148,
  'random_damage_mean': 0.687,
  'damage_lift': 0.11985669781931474,
  'policy_stars_mean': 2.3,
  'random_stars_mean': 1.414},
 'hard': {'policy_score_mean': 0.4881967963386728,
  'random_score_mean': 0.308,
  'score_lift': 0.1801967963386728,
  'policy_damage_mean': 0.2881967963386728,
  'random_damage_mean': 0.274,
  'damage_lift': 0.014196796338672768,
  'policy_stars_mean': 0.2,
  'random_stars_mean': 0.034}}

## Decision Rubric

Use this after the cells above finish.

- If medium holdout has `score_mean >= 3.0`, `stars_mean >= 2.0`, and `p_damage_ge_90 >= 0.50`, the run is a real learnability proof.
- If cross-profile hard beats hard random by a clear margin but remains weak, use curriculum next: medium-heavy mix with hard gradually increased.
- If medium is strong but hard is near random, do not spend a long hard-only run yet. Add curriculum and inspect visual rollouts first.
- If medium regresses versus the 20-episode in-training eval, rerun with a second seed before changing the algorithm.

Throughput remains the main engineering blocker. Before a long hard/curriculum run, the highest-leverage implementation work is: flattened training observations, vectorized deploy masks, training fast-mode with viewer events disabled, and then a custom mask-returning rollout collector if needed.

In [ ]:
manifest = {
    "run_id": RUN_ID,
    "checkpoint_path": str(CHECKPOINT_PATH),
    "device": DEVICE,
    "deterministic": DETERMINISTIC,
    "army_composition": ARMY_COMPOSITION,
    "max_buildings": MAX_BUILDINGS,
    "medium_holdout": medium_summary if "medium_summary" in globals() else None,
    "cross_profile": cross_profile_summaries if "cross_profile_summaries" in globals() else None,
    "artifacts": str(ARTIFACT_DIR),
}
manifest_path = ARTIFACT_DIR / "manifest.json"
with manifest_path.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print(f"wrote {manifest_path}")
manifest